# 03. IsolationForest, 이전 LOT으로 다음 LOT 살펴보기

CSV 이름만 바꾸면 같은 실험을 다시 실행함. 각 시점의 센서값, 과거 60초에서 만든 B 입력, 강종으로 그 시점의 이상 후보를 찾음.

| 방법 | 학습에 넣는 LOT | 확인할 LOT |
| --- | --- | --- |
| 고정 9/2/3 | 앞 9개 | 검증 2개, 시험 3개 |
| 이전 동일 강종 | 해당 LOT보다 먼저 끝난 같은 강종 전부 | 해당 LOT |
| 이전 모든 강종 | 해당 LOT보다 먼저 끝난 LOT 전부 | 해당 LOT |

`LABEL`은 결과 비교에만 사용함. 학습 표본은 라벨과 관계없이 고름. 센서별 기준은 **Train 전체 점수의 95% 지점**으로 고정함. 21개 중 하나라도 넘으면 행을 후보로 표시하므로 **행 후보율이 5%라는 뜻은 아님**.

In [ ]:
DATA_FILE = "cal01_base_v1.csv"  # 바꿀 입력 파일 이름
RUN_FIRST_LOT = False  # 001 내부를 실제 시간 60/20/20으로 나누는 추가 실험

import os
import json
import hashlib
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import sklearn
from matplotlib import font_manager, rcParams
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import display
from sklearn.ensemble import IsolationForest

ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, "notebooks")):
    ROOT = os.path.dirname(ROOT)
if not os.path.isdir(os.path.join(ROOT, "notebooks")):
    ROOT = os.path.dirname(ROOT)

ENC = "utf-8-sig"
PROCESSED_DIR = os.path.join(ROOT, "data", "processed")
BASE_CSV = os.path.join(PROCESSED_DIR, DATA_FILE)
with open(BASE_CSV, "rb") as source_file:
    INPUT_SHA = hashlib.sha256(source_file.read()).hexdigest()

RUN_ID = os.path.splitext(DATA_FILE)[0] + "__" + INPUT_SHA[:8] + "__" + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
OUT_DIR = os.path.join(ROOT, "data", "processed", "sayletskin", "isolation_forest_ab", RUN_ID)
FIG_DIR = os.path.join(ROOT, "figures", "sayletskin", "isolation_forest_ab", RUN_ID)
os.makedirs(OUT_DIR)
os.makedirs(FIG_DIR)

font_names = []
for installed_font in font_manager.fontManager.ttflist:
    font_names.append(installed_font.name)
for font_name in ["Malgun Gothic", "AppleGothic", "NanumGothic", "NanumBarunGothic", "Noto Sans CJK KR"]:
    if font_name in font_names:
        rcParams["font.family"] = font_name
        break
rcParams["axes.unicode_minus"] = False
rcParams["figure.dpi"] = 110

SEED = 42
N_ESTIMATORS = 60
MAX_FIT_ROWS = 12000
MAX_SAMPLES = 256
MAX_GAP_SECONDS = 15
MIN_PAST_ROWS = 10
TRAIN_QUANTILE = 0.95

TRAIN_LOTS = ["240001", "240024", "240037", "240038", "240044", "240058", "240061", "240069", "240091"]
VALID_LOTS = ["240108", "240114"]
TEST_LOTS = ["240117", "240121", "240133"]
LOTS = TRAIN_LOTS + VALID_LOTS + TEST_LOTS
SENSORS = ["CLN-IN", "TC-Z1", "TC-Z2", "OP-Z1", "OP-Z2", "TC-Z3", "TC-Z4", "TC-Z5", "TC-Z6", "OP-Z3", "OP-Z4", "OP-Z5", "OP-Z6", "CP-Z4", "TC-Z7", "TC-Z8", "TC-Z9", "OP-Z7", "OP-Z8", "TC-OUT1", "TC-OUT2"]
META = ["ROW_ID", "LOT_NO", "MEAS_DT", "GRADE", "EXCLUDED", "LABEL"]

print("입력:", BASE_CSV)
print("결과:", OUT_DIR)
print("그림:", FIG_DIR)

## 1. 입력과 과거 60초

현재값, 초당 변화량, 직전 60초 평균과의 차이, 직전 60초 표준편차를 직접 계산함. 현재 시점은 평균과 표준편차에서 뺌.

LOT 변경, 15초 초과 공백, 제외 여부 변경에서 과거 기록을 끊음. 제외한 값은 학습과 과거값 계산에 넣지 않음. 결측은 채우지 않고 **21개 센서의 입력이 모두 있는 같은 행**을 비교함. CSV의 기존 파생변수와 `MODEL_ELIGIBLE`은 사용하지 않음.

In [ ]:
df = pd.read_csv(BASE_CSV, encoding=ENC, usecols=META + SENSORS, dtype={"LOT_NO": "string", "GRADE": "string"}, low_memory=False)
df["MEAS_DT"] = pd.to_datetime(df["MEAS_DT"], errors="raise")
df = df.sort_values(["LOT_NO", "MEAS_DT"], kind="stable").reset_index(drop=True)
# 제외 여부가 비어 있거나 뜻을 알 수 없으면 학습에서 뺌.
df["EXCLUDED"] = ~df["EXCLUDED"].astype("string").str.strip().str.lower().isin(["false", "0", "0.0"])
df["LABEL"] = pd.to_numeric(df["LABEL"], errors="coerce")
df["GRADE"] = df["GRADE"].fillna("미기록")
for sensor in SENSORS:
    df[sensor] = pd.to_numeric(df[sensor], errors="coerce").replace([np.inf, -np.inf], np.nan)

seconds = df["MEAS_DT"].diff().dt.total_seconds()
breaks = df["LOT_NO"].ne(df["LOT_NO"].shift()).fillna(True)
breaks = breaks | seconds.gt(MAX_GAP_SECONDS) | seconds.le(0)
breaks = breaks | df["EXCLUDED"].ne(df["EXCLUDED"].shift())
df["SEGMENT"] = breaks.cumsum()
features = {}
common = ~df["EXCLUDED"].to_numpy()

for sensor in SENSORS:
    feature = pd.DataFrame(np.nan, index=df.index, columns=["VALUE", "DIFF_PER_SEC", "DEV_60S", "STD_60S"], dtype="float64")
    for segment, block in df.loc[~df["EXCLUDED"]].groupby("SEGMENT", sort=False):
        values = pd.Series(block[sensor].to_numpy(), index=pd.DatetimeIndex(block["MEAS_DT"]))
        elapsed = values.index.to_series().diff().dt.total_seconds()
        past = values.rolling("60s", closed="left", min_periods=MIN_PAST_ROWS)
        feature.loc[block.index, "VALUE"] = values.to_numpy()
        feature.loc[block.index, "DIFF_PER_SEC"] = values.diff().div(elapsed).to_numpy()
        feature.loc[block.index, "DEV_60S"] = values.sub(past.mean()).to_numpy()
        feature.loc[block.index, "STD_60S"] = past.std().to_numpy()

    feature = feature.replace([np.inf, -np.inf], np.nan).astype("float32")
    features[sensor] = feature
    common = common & np.isfinite(feature.to_numpy()).all(axis=1)

df["COMMON_VALID"] = common
lot_info = df.groupby("LOT_NO", sort=False).agg(GRADE=("GRADE", "first"), START=("MEAS_DT", "min"), END=("MEAS_DT", "max"), TOTAL_ROWS=("ROW_ID", "size"), COMMON_ROWS=("COMMON_VALID", "sum"))
display(lot_info.rename(columns={"GRADE": "강종", "START": "관측 시작", "END": "관측 끝", "TOTAL_ROWS": "전체 행", "COMMON_ROWS": "계산 가능한 행"}))

## 2. 시간 순서 고정

이전 기록이 없으면 해당 실험을 생략함. 117의 강종은 이전에 없어 동일 강종 실험을 할 수 없음.

001 추가 실험을 켜면 행 개수가 아닌 실제 관측 시간의 60%, 80% 지점에서 나눔. 시험 중 이미 관측한 직전 60초는 현재 시점의 입력으로 사용함.

In [ ]:
jobs = [{"case": "fixed_9_2_3", "method": "fixed_9_2_3", "roles": {"train": TRAIN_LOTS, "valid": VALID_LOTS, "test": TEST_LOTS}}]

for method in ["same_grade_history", "all_history"]:
    for lot in LOTS:
        history = []
        if lot in lot_info.index:
            previous = lot_info.loc[lot_info["END"].lt(lot_info.loc[lot, "START"])].sort_values("START")
            if method == "same_grade_history":
                previous = previous.loc[previous["GRADE"].eq(lot_info.loc[lot, "GRADE"])]
            history = previous.index.tolist()
        jobs.append({"case": method + "_" + lot, "method": method, "roles": {"train": history, "test": [lot]}})

if RUN_FIRST_LOT and "240001" in lot_info.index:
    first_start = lot_info.loc["240001", "START"]
    first_span = lot_info.loc["240001", "END"] - first_start
    jobs.append({
        "case": "first_lot_60_20_20",
        "method": "first_lot_60_20_20",
        "roles": {"train": ["240001"], "valid": ["240001"], "test": ["240001"]},
        "cuts": [first_start + first_span * 0.6, first_start + first_span * 0.8],
    })

plan_rows = []
for job in jobs:
    job["indices"] = {}
    for role, lots in job["roles"].items():
        selected = df["LOT_NO"].isin(lots)
        if "cuts" in job:
            if role == "train":
                selected = selected & df["MEAS_DT"].lt(job["cuts"][0])
            elif role == "valid":
                selected = selected & df["MEAS_DT"].ge(job["cuts"][0]) & df["MEAS_DT"].lt(job["cuts"][1])
            else:
                selected = selected & df["MEAS_DT"].ge(job["cuts"][1])
        job["indices"][role] = df.index[selected].to_numpy()

    train_index = job["indices"]["train"]
    job["train_index"] = train_index[common[train_index]]
    job["status"] = "실행" if len(job["train_index"]) else "생략"
    job["reason"] = ""
    if not job["roles"]["train"]:
        job["reason"] = "이전 동일 강종 기록 없음" if job["method"] == "same_grade_history" else "이전 LOT 기록 없음"
    elif not len(job["train_index"]):
        job["reason"] = "학습에 사용할 공통 유효행 없음"
    if job["method"] == "fixed_9_2_3":
        missing_lots = []
        for required_lot in LOTS:
            if required_lot not in lot_info.index:
                missing_lots.append(required_lot)
        if missing_lots:
            job["train_index"] = np.array([], dtype=int)
            job["status"] = "생략"
            job["reason"] = "고정 9/2/3에 필요한 LOT 없음: " + ",".join(missing_lots)

    for role, lots in job["roles"].items():
        for lot in lots:
            index = job["indices"][role]
            index = index[df.loc[index, "LOT_NO"].eq(lot).to_numpy(dtype=bool)]
            n_common = int(common[index].sum())
            status, reason = job["status"], job["reason"]
            if not len(index):
                status, reason = "생략", "CSV에 해당 LOT 또는 시간 구간 없음"
            elif not n_common:
                status, reason = "생략", "계산에 사용할 공통 유효행 없음"
            plan_rows.append({
                "CASE": job["case"],
                "METHOD": job["method"],
                "ROLE": role,
                "LOT_NO": lot,
                "GRADE": lot_info.loc[lot, "GRADE"] if lot in lot_info.index else "",
                "START": df.loc[index, "MEAS_DT"].min(),
                "END": df.loc[index, "MEAS_DT"].max(),
                "STATUS": status,
                "REASON": reason,
                "TOTAL_ROWS": len(index),
                "COMMON_ROWS": n_common,
                "TRAIN_LOTS": ",".join(job["roles"]["train"]),
            })

split_plan = pd.DataFrame(plan_rows)
split_plan.to_csv(os.path.join(OUT_DIR, "split_plan.csv"), index=False, encoding=ENC)
display(split_plan.loc[split_plan["ROLE"].ne("train"), ["METHOD", "ROLE", "LOT_NO", "TRAIN_LOTS", "COMMON_ROWS", "STATUS", "REASON"]].rename(columns={
    "METHOD": "방법",
    "ROLE": "역할",
    "LOT_NO": "LOT",
    "TRAIN_LOTS": "학습 LOT",
    "COMMON_ROWS": "계산 가능한 행",
    "STATUS": "상태",
    "REASON": "이유",
}))

## 3. 학습과 판정

Train 안에서 LOT별로 같은 표본 수를 배정해 최대 12,000행으로 학습함. 기록이 적은 LOT는 있는 행만 사용함. 기준 점수는 이 표본이 아닌 **Train 전체 유효행**으로 계산함. 점수가 기준보다 클 때만 후보 1임.

강종 열도 Train에 있는 종류만 만듦. 처음 보는 강종은 별도로 표시함. Valid/Test의 값과 라벨은 학습이나 기준 점수를 정하는 데 사용하지 않음.

In [ ]:
threshold_rows = []
fit_parts = []
prediction_parts = []
score_columns = []
flag_columns = []
for sensor in SENSORS:
    score_columns.append(sensor + "__SCORE")
    flag_columns.append(sensor + "__FLAG")

for job in jobs:
    train_index = job["train_index"]
    evaluation_index = np.array([], dtype=int)
    role_values = []
    for role in ["valid", "test"]:
        if role in job["indices"]:
            index = job["indices"][role]
            evaluation_index = np.concatenate([evaluation_index, index])
            role_values.extend([role] * len(index))

    result = df.loc[evaluation_index, META + ["COMMON_VALID"]].copy()
    result["CASE"] = job["case"]
    result["METHOD"] = job["method"]
    result["ROLE"] = role_values
    result["STATUS"] = job["status"]
    result["REASON"] = job["reason"]
    result["SENSOR_COUNT"] = len(SENSORS)
    result["FLAG_COUNT"] = np.nan
    result["ROW_PRED"] = np.nan
    result["UNSEEN_GRADE"] = pd.NA
    for column in score_columns + flag_columns:
        result[column] = np.nan

    if not len(train_index):
        prediction_parts.append(result)
        continue

    train_lots = df.loc[train_index, "LOT_NO"].unique()
    per_lot = MAX_FIT_ROWS // len(train_lots)
    picked = []
    rng = np.random.default_rng(SEED)
    for lot in train_lots:
        index = train_index[df.loc[train_index, "LOT_NO"].eq(lot).to_numpy(dtype=bool)]
        picked.extend(rng.choice(index, size=min(per_lot, len(index)), replace=False).tolist())
    fit_index = np.sort(np.array(picked, dtype=int))
    fit_record = df.loc[fit_index, ["ROW_ID", "LOT_NO", "MEAS_DT", "GRADE"]].copy()
    fit_record["CASE"] = job["case"]
    fit_record["METHOD"] = job["method"]
    fit_parts.append(fit_record)

    grade_train = pd.get_dummies(df.loc[train_index, "GRADE"], prefix="GRADE", dtype="float32")
    grade_columns = grade_train.columns.tolist()
    score_index = evaluation_index[common[evaluation_index]]
    grade_evaluation = pd.get_dummies(df.loc[score_index, "GRADE"], prefix="GRADE", dtype="float32")
    grade_evaluation = grade_evaluation.reindex(columns=grade_columns, fill_value=0)
    result["UNSEEN_GRADE"] = ~result["GRADE"].isin(df.loc[train_index, "GRADE"].unique())

    for sensor in SENSORS:
        x_train = pd.concat([features[sensor].loc[train_index], grade_train], axis=1)
        model = IsolationForest(n_estimators=N_ESTIMATORS, max_samples=min(MAX_SAMPLES, len(fit_index)), contamination="auto", random_state=SEED, n_jobs=2)
        model.fit(x_train.loc[fit_index].to_numpy(dtype="float32"))
        train_score = -model.score_samples(x_train.to_numpy(dtype="float32"))
        threshold = float(np.quantile(train_score, TRAIN_QUANTILE))
        threshold_rows.append({
            "CASE": job["case"],
            "METHOD": job["method"],
            "SENSOR": sensor,
            "THRESHOLD": threshold,
            "TRAIN_ROWS": len(train_index),
            "FIT_ROWS": len(fit_index),
            "TRAIN_LOTS": ",".join(train_lots),
            "GRADE_COLUMNS": ",".join(grade_columns),
        })

        if len(score_index):
            x_evaluation = pd.concat([features[sensor].loc[score_index], grade_evaluation], axis=1)
            score = -model.score_samples(x_evaluation.to_numpy(dtype="float32"))
            result.loc[score_index, sensor + "__SCORE"] = score
            result.loc[score_index, sensor + "__FLAG"] = (score > threshold).astype("int8")

    if len(score_index):
        result.loc[score_index, "FLAG_COUNT"] = result.loc[score_index, flag_columns].sum(axis=1)
        result.loc[score_index, "ROW_PRED"] = result.loc[score_index, "FLAG_COUNT"].gt(0).astype("int8")
    prediction_parts.append(result)
    print(job["case"], "| Train 전체", len(train_index), "| fit 표본", len(fit_index), "| Valid/Test 점수", len(score_index))

predictions = pd.concat(prediction_parts, ignore_index=True)
thresholds = pd.DataFrame(threshold_rows)
fit_rows = pd.concat(fit_parts, ignore_index=True) if fit_parts else pd.DataFrame(columns=["ROW_ID", "LOT_NO", "MEAS_DT", "GRADE", "CASE", "METHOD"])
predictions.to_csv(os.path.join(OUT_DIR, "predictions.csv"), index=False, encoding=ENC, float_format="%.17g")
thresholds.to_csv(os.path.join(OUT_DIR, "thresholds.csv"), index=False, encoding=ENC, float_format="%.17g")
fit_rows.to_csv(os.path.join(OUT_DIR, "fit_rows.csv"), index=False, encoding=ENC)

## 4. 같은 행의 기존 라벨과 비교

라벨이 비어 있으면 모델 점수만 남기고 지표 계산에서는 뺌. 후보율도 라벨이 있는 평가행 기준임. 분모가 0인 지표는 빈칸으로 남김.

21개 센서 각각의 후보 수와 비율은 `sensor_summary.csv`에 저장함. 행에 붙은 라벨을 센서별 정답으로 사용하지 않음.

In [ ]:
metric_rows = []
for (case, method, role, lot), block in predictions.groupby(["CASE", "METHOD", "ROLE", "LOT_NO"], sort=False):
    scored = block.loc[block["ROW_PRED"].notna()]
    evaluated = scored.loc[scored["LABEL"].isin([0, 1])]
    y = evaluated["LABEL"]
    pred = evaluated["ROW_PRED"]
    tp = int((y.eq(1) & pred.eq(1)).sum())
    fp = int((y.eq(0) & pred.eq(1)).sum())
    fn = int((y.eq(1) & pred.eq(0)).sum())
    tn = int((y.eq(0) & pred.eq(0)).sum())
    reason = block["REASON"].iloc[0]
    if not len(scored) and not reason:
        reason = "계산 가능한 공통 유효행 없음"
    metric_rows.append({
        "CASE": case,
        "METHOD": method,
        "ROLE": role,
        "LOT_NO": lot,
        "GRADE": block["GRADE"].iloc[0],
        "N_ROWS": len(scored),
        "N_EVALUATED": len(evaluated),
        "LABEL1": int(y.eq(1).sum()),
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
        "PRECISION": tp / (tp + fp) if tp + fp else np.nan,
        "RECALL": tp / (tp + fn) if tp + fn else np.nan,
        "F1": 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else np.nan,
        "DETECTION_RATE": float(pred.eq(1).mean()) if len(evaluated) else np.nan,
        "UNSEEN_GRADE": bool(block["UNSEEN_GRADE"].fillna(False).any()) if len(scored) else pd.NA,
        "STATUS": "실행" if len(scored) else "생략",
        "REASON": reason,
    })

metrics = pd.DataFrame(metric_rows)
metrics.to_csv(os.path.join(OUT_DIR, "metrics.csv"), index=False, encoding=ENC, float_format="%.7g")
method_names = {"fixed_9_2_3": "고정 9/2/3", "same_grade_history": "이전 동일 강종", "all_history": "이전 모든 강종", "first_lot_60_20_20": "001 시간 60/20/20"}
show_columns = ["METHOD", "ROLE", "LOT_NO", "N_ROWS", "N_EVALUATED", "LABEL1", "TP", "FP", "FN", "TN", "PRECISION", "RECALL", "F1", "DETECTION_RATE", "UNSEEN_GRADE", "REASON"]
show_names = {
    "METHOD": "방법",
    "ROLE": "역할",
    "LOT_NO": "LOT",
    "N_ROWS": "점수 계산 행",
    "N_EVALUATED": "라벨 비교 행",
    "LABEL1": "기존 라벨 1",
    "TP": "찾은 라벨 1",
    "FP": "기존 0인 후보",
    "FN": "놓친 라벨 1",
    "TN": "기존 0, 후보 아님",
    "PRECISION": "Precision",
    "RECALL": "Recall",
    "DETECTION_RATE": "행 후보율",
    "UNSEEN_GRADE": "처음 보는 강종",
    "REASON": "생략 이유",
}
shown = metrics[show_columns].replace({"METHOD": method_names}).rename(columns=show_names)
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(shown.round(4))

print("117, 121, 133의 같은 시험 행 비교")
comparison = metrics.loc[metrics["ROLE"].eq("test") & metrics["LOT_NO"].isin(TEST_LOTS), show_columns]
display(comparison.replace({"METHOD": method_names}).rename(columns=show_names).round(4))

sensor_rows = []
for (case, method, role, lot), block in predictions.groupby(["CASE", "METHOD", "ROLE", "LOT_NO"], sort=False):
    for sensor in SENSORS:
        flag = block[sensor + "__FLAG"]
        sensor_rows.append({
            "CASE": case,
            "METHOD": method,
            "ROLE": role,
            "LOT_NO": lot,
            "SENSOR": sensor,
            "SCORED_ROWS": int(flag.notna().sum()),
            "CANDIDATE_ROWS": int(flag.eq(1).sum()),
            "CANDIDATE_RATE": flag.mean(),
        })
sensor_summary = pd.DataFrame(sensor_rows)
sensor_summary.to_csv(os.path.join(OUT_DIR, "sensor_summary.csv"), index=False, encoding=ENC)


## 5. LOT별 결과표 바로 아래 그래프

위 두 센서는 모델이 후보로 잡은 비율이 높은 순서로 고름. 파란 선은 원자료임.

아래 칸은 **1분 안에서 이상 후보 1인 기록의 비율**임. 진한 노란색일수록 비율이 높고, 회색은 제외 또는 미계산임. 표와 같은 평가행으로 비율을 계산하며, 모델과 지표는 원래 행 단위를 유지함. 센서선은 기록 공백과 결측을 이어 붙이지 않음.

In [ ]:
for lot in LOTS:
    lot_predictions = predictions.loc[predictions["LOT_NO"].eq(lot)]
    if not lot_predictions["ROW_PRED"].notna().any():
        continue

    lot_metrics = metrics.loc[metrics["LOT_NO"].eq(lot), show_columns]
    print("LOT", lot)
    display(lot_metrics.replace({"METHOD": method_names}).rename(columns=show_names).round(4))

    sensor_rates = lot_predictions.groupby("METHOD")[flag_columns].mean().max(axis=0).sort_values(ascending=False)
    plot_sensors = []
    for column in sensor_rates.head(2).index:
        plot_sensors.append(column.removesuffix("__FLAG"))
    selected_sensors = pd.DataFrame({"그림에서 볼 센서": plot_sensors, "방법별 후보율 중 가장 높은 값": sensor_rates.head(2).to_numpy()})
    display(selected_sensors.round(4))

    raw = df.loc[df["LOT_NO"].eq(lot)].copy()
    raw["DRAW_SEGMENT"] = raw["MEAS_DT"].diff().dt.total_seconds().gt(MAX_GAP_SECONDS).cumsum()
    methods = []
    for method in method_names:
        if lot_predictions["METHOD"].eq(method).any():
            methods.append(method)
    fig, axes = plt.subplots(3, 1, figsize=(14, 7.5), sharex=True, gridspec_kw={"height_ratios": [2, 2, 2.4]})
    for axis, sensor in zip(axes[:2], plot_sensors):
        for segment, block in raw.groupby("DRAW_SEGMENT", sort=False):
            axis.plot(block["MEAS_DT"], block[sensor], color="#396788", linewidth=0.7)
        unit = "기록값"
        if sensor.startswith("TC-"):
            unit = "℃"
        elif sensor.startswith("OP-"):
            unit = "%"
        axis.set_ylabel(sensor + " (" + unit + ")")
        axis.grid(alpha=0.15)

    excluded_groups = raw["EXCLUDED"].ne(raw["EXCLUDED"].shift()).cumsum()
    for group, block in raw.groupby(excluded_groups, sort=False):
        if block["EXCLUDED"].iloc[0]:
            for axis in axes:
                axis.axvspan(block["MEAS_DT"].iloc[0], block["MEAS_DT"].iloc[-1] + pd.Timedelta(seconds=1), color="#c4c7cc", alpha=0.35)

    minute_edges = pd.date_range(raw["MEAS_DT"].min().floor("min"), raw["MEAS_DT"].max().floor("min") + pd.Timedelta(minutes=1), freq="min")
    minute_index = minute_edges[:-1]
    common_ids = lot_predictions.loc[lot_predictions["ROW_PRED"].notna(), "ROW_ID"].unique()
    labeled = raw.loc[raw["ROW_ID"].isin(common_ids) & raw["LABEL"].isin([0, 1])]
    label_rate = labeled.groupby(labeled["MEAS_DT"].dt.floor("min"))["LABEL"].mean()
    minute_rates = [label_rate.reindex(minute_index).to_numpy()]
    tick_labels = ["기존 라벨"]

    for method in methods:
        block = lot_predictions.loc[lot_predictions["METHOD"].eq(method)]
        evaluated = block.loc[block["LABEL"].isin([0, 1]) & block["ROW_PRED"].notna()]
        rate = evaluated.groupby(evaluated["MEAS_DT"].dt.floor("min"))["ROW_PRED"].mean()
        minute_rates.append(rate.reindex(minute_index).to_numpy())
        roles = ",".join(block["ROLE"].unique())
        suffix = " (계산 불가)" if evaluated.empty else " (" + roles + ")"
        tick_labels.append(method_names[method] + suffix)

    colors = LinearSegmentedColormap.from_list("candidate_rate", ["#fffbee", "#f4c24a", "#a56800"])
    colors.set_bad("#e8ebee")
    heat = axes[2].pcolormesh(
        mdates.date2num(minute_edges), np.arange(len(tick_labels) + 1) - 0.5,
        np.array(minute_rates), cmap=colors, vmin=0, vmax=1, shading="flat",
    )
    axes[2].set_yticks(range(len(tick_labels)), tick_labels)
    axes[2].set_ylim(len(tick_labels) - 0.5, -0.5)
    axes[2].xaxis.set_major_formatter(mdates.DateFormatter("%m/%d\n%H:%M"))
    axes[2].set_xlabel(str(raw["MEAS_DT"].iloc[0].year) + "년 실제 날짜와 시각")
    fig.suptitle(DATA_FILE + " | LOT " + lot + " | 센서 기록과 이상 후보를 같은 시간에 비교")
    fig.subplots_adjust(left=0.18, right=0.985, top=0.93, bottom=0.23, hspace=0.12)
    color_axis = fig.add_axes([0.38, 0.09, 0.5, 0.02])
    colorbar = fig.colorbar(heat, cax=color_axis, orientation="horizontal", ticks=[0, 0.25, 0.5, 0.75, 1])
    colorbar.ax.set_xticklabels(["0%", "25%", "50%", "75%", "100%"])
    colorbar.set_label("1분 안에서 이상 후보 1인 기록의 비율")
    fig.text(0.18, 0.045, "회색: 학습 제외 또는 미계산", fontsize=9, color="#68717a")
    fig.savefig(os.path.join(FIG_DIR, "if_compare_" + lot + ".png"), dpi=130, bbox_inches="tight")
    plt.show()
    plt.close(fig)

In [ ]:
settings = {
    "input_file": DATA_FILE,
    "input_sha256": INPUT_SHA,
    "run_id": RUN_ID,
    "sklearn_version": sklearn.__version__,
    "seed": SEED,
    "n_estimators": N_ESTIMATORS,
    "max_fit_rows": MAX_FIT_ROWS,
    "max_samples": MAX_SAMPLES,
    "n_jobs": 2,
    "contamination": "auto",
    "train_score_quantile": TRAIN_QUANTILE,
    "flag_rule": "score > threshold; row prediction = any of 21 sensor flags",
    "sensors": SENSORS,
    "features_per_sensor": ["VALUE", "DIFF_PER_SEC", "DEV_60S", "STD_60S"],
    "rolling": {
        "seconds": 60,
        "closed": "left",
        "min_periods": MIN_PAST_ROWS,
        "std_ddof": 1,
        "reset_gap_seconds": MAX_GAP_SECONDS,
        "reset_on_lot_or_excluded_change": True,
    },
    "excluded_policy": "excluded rows are absent from fit, scoring and feature history",
    "validity_policy": "all 21 sensors and four features finite; no imputation; MODEL_ELIGIBLE unused",
    "grade_policy": "get_dummies from Train only; evaluation reindexed with fill_value=0",
    "fit_sampling": "equal quota by available Train LOT, at most 12000 rows, seed 42",
    "fixed_roles": {"train": TRAIN_LOTS, "valid": VALID_LOTS, "test": TEST_LOTS},
    "history_policy": "only LOTs ending before target LOT starts",
    "run_first_lot": RUN_FIRST_LOT,
    "first_lot_policy": "actual observed time 60/20/20; train cutoff exclusive",
    "plot_aggregation": "per-minute fraction of candidate rows, evaluated rows only; models and metrics remain at source-row resolution",
    "prediction_export": "Valid/Test rows only; blank scores for excluded, invalid and unavailable cases",
    "label_use": "metrics only; missing labels omitted; v2 labels use whole-LOT statistics",
    "feature_exclusions": ["LABEL", "ROW_ID", "LOT_NO", "existing derived columns", "maintenance and domain decision columns"],
    "output_csvs": ["split_plan.csv", "metrics.csv", "thresholds.csv", "predictions.csv", "fit_rows.csv", "sensor_summary.csv"]
}
with open(os.path.join(OUT_DIR, "settings.json"), "w", encoding="utf-8") as settings_file:
    json.dump(settings, settings_file, ensure_ascii=False, indent=2)
print("저장 완료:", OUT_DIR)
print("그림:", FIG_DIR)

## 결과를 읽을 때

- 기존 라벨은 고장 정답이 아닌 이상 후보임. v2 라벨은 각 LOT 전체의 통계를 사용했으므로 실제 미래 고장 성능으로 읽지 않음.
- 모델 학습과 기준 점수는 Train만 사용함. Valid는 보고만 하며, 95% 기준을 결과에 맞춰 바꾸지 않음. v1의 Valid에는 라벨 1이 없음.
- 이전 모든 강종 방식은 121을 볼 때 117, 133을 볼 때 121까지 센서 기록이 추가됨. 고정 9/2/3과 학습 이력이 다름.
- 입력 B의 과거값과 제외 처리를 바로잡은 별도 실험임. 기존 PR #84의 전 LOT 학습 결과와 구분함.

CSV 이름, 해시, 실행 시각별 폴더에 저장하므로 이전 결과와 원본 CSV는 덮어쓰지 않음. `predictions.csv`는 Valid/Test만 저장하며, Train 전체 행수와 fit 표본은 `thresholds.csv`, `fit_rows.csv`에서 확인함.